In [1]:
import os
import glob
import numpy as np
import pandas as pd

In [2]:
FOLDER = os.environ.get("UNSW_FOLDER", r"C:\Users\Lenovo\Documents\pbl\dataset one")
OUT_BASE = os.path.join(FOLDER, "unsw_clean")

In [3]:
CATEGORY_MAP = {
    "normal": "Normal", "generic": "Generic", "exploits": "Exploits",
    "fuzzers": "Fuzzers", "dos": "DoS", "reconnaissance": "Reconnaissance",
    "analysis": "Analysis", "backdoor": "Backdoor", "backdoors": "Backdoor",
    "shellcode": "Shellcode", "worms": "Worms",
}

In [4]:
def load_column_names(folder):
    feat = pd.read_csv(os.path.join(folder, "NUSW-NB15_features.csv"), encoding="latin1")
    names = [n.strip().replace(" ", "") for n in feat["Name"]]   # fixes 'ct_src_ ltm'
    assert len(names) == 49, f"expected 49 names, got {len(names)}"
    return names

In [5]:
def load_raw(folder, names):
    files = sorted(glob.glob(os.path.join(folder, "UNSW-NB15_[1-4].csv")))
    assert files, f"no UNSW-NB15_[1-4].csv files found in {folder}"
    if len(files) != 4:
        print(f"WARNING: found {len(files)} of 4 raw files")
    parts = []
    for f in files:
        part = pd.read_csv(f, header=None, names=names, low_memory=False)
        print(f"loaded {os.path.basename(f)}: {part.shape}")
        parts.append(part)
    df = pd.concat(parts, ignore_index=True)
    print("combined shape:", df.shape)
    return df

In [6]:
def _hex_or_nan(v):
    try:
        return int(v, 16)
    except ValueError:
        return np.nan

In [7]:
def to_number(series):
    """Strip spaces, convert hex strings like '0x000b', everything else via to_numeric."""
    s = series.astype(str).str.strip()
    is_hex = s.str.lower().str.startswith("0x")
    out = pd.to_numeric(s.where(~is_hex), errors="coerce").astype("float64")
    if is_hex.any():
        out[is_hex] = [_hex_or_nan(v) for v in s[is_hex]]
    return out

In [8]:
def clean_ports(df):
    for col in ["sport", "dsport"]:
        before = df[col].astype(str).str.strip()
        num = to_number(df[col])
        bad = num.isna()
        if bad.any():
            print(f"{col}: {bad.sum()} values not numeric -> -1. examples:",
                  before[bad].value_counts().head(5).to_dict())
        df[col] = num.fillna(-1).astype("int32")       # -1 = unknown (ports are not model inputs)
    return df

In [9]:
def clean_ftp_http_columns(df):
    # ct_ftp_cmd: object column with blank cells
    num = to_number(df["ct_ftp_cmd"])
    print(f"ct_ftp_cmd: {num.isna().sum()} blank/non-numeric -> 0")
    df["ct_ftp_cmd"] = num.fillna(0).astype("int32")
 
    # two columns that are NaN on non-HTTP / non-FTP rows
    is_http = df["service"].astype(str).str.strip() == "http"
    for col in ["ct_flw_http_mthd", "is_ftp_login"]:
        df[col] = to_number(df[col])
        miss = df[col].isna()
        print(f"{col}: NaN in {miss[is_http].mean():.1%} of http rows, "
              f"{miss[~is_http].mean():.1%} of non-http rows")
        df[col + "_was_missing"] = miss.astype("int8")  # keep the information
        df[col] = df[col].fillna(0).astype("int32")
    print("is_ftp_login values:", df["is_ftp_login"].value_counts().to_dict())
    return df

In [10]:
def clean_labels(df):
    df["Label"] = pd.to_numeric(df["Label"], errors="coerce")
    assert df["Label"].isin([0, 1]).all(), "Label has values other than 0/1"
    df["Label"] = df["Label"].astype("int8")
 
    raw = df["attack_cat"].astype("string").str.strip().str.lower()
    cat = raw.map(CATEGORY_MAP)
    unmapped = cat.isna() & raw.notna() & (raw != "")
    if unmapped.any():
        print("WARNING unmapped attack_cat values:", raw[unmapped].value_counts().to_dict())
    blank = cat.isna() & ~unmapped
    cat[blank & (df["Label"] == 0)] = "Normal"
    cat[blank & (df["Label"] == 1)] = "Unknown"
    cat[unmapped] = "Unknown"
    df["attack_cat"] = cat.astype("string")
 
    mismatch = ((df["Label"] == 1) & (df["attack_cat"] == "Normal")) | \
               ((df["Label"] == 0) & (df["attack_cat"] != "Normal"))
    print(f"Label vs attack_cat mismatches: {mismatch.sum()}")
    print(df["attack_cat"].value_counts().to_dict())
    return df

In [11]:
def clean_time_and_sort(df):
    for col in ["Stime", "Ltime"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    bad = df["Stime"].isna()
    if bad.any():
        print(f"dropping {bad.sum()} rows with bad Stime")
        df = df[~bad]
    df["Stime"] = pd.to_datetime(df["Stime"], unit="s")
    df["Ltime"] = pd.to_datetime(df["Ltime"], unit="s")
    df = df.sort_values("Stime", kind="mergesort").reset_index(drop=True)
    print("Stime range:", df["Stime"].min(), "->", df["Stime"].max())
    return df

In [12]:
def tidy_text_columns(df):
    for col in ["srcip", "dstip", "proto", "state", "service"]:
        df[col] = df[col].astype(str).str.strip().astype("category")
    return df
 
 
def save(df):
    try:
        df.to_parquet(OUT_BASE + ".parquet")
        print("saved", OUT_BASE + ".parquet")
    except Exception as e:                         # pyarrow missing, etc.
        df.to_pickle(OUT_BASE + ".pkl")
        print(f"parquet failed ({type(e).__name__}); saved", OUT_BASE + ".pkl")

In [13]:
def main():
    names = load_column_names(FOLDER)
    df = load_raw(FOLDER, names)
    assert not (df["srcip"] == "srcip").any(), "stray header row found"
    df = clean_ports(df)
    df = clean_ftp_http_columns(df)
    df = clean_labels(df)
    df = clean_time_and_sort(df)
    df = tidy_text_columns(df)
 
    print("\nrows and attack share per day:")
    print(df.groupby(df["Stime"].dt.date)["Label"].agg(rows="size", attack_share="mean"))
    print("\nnull counts (non-zero only):")
    nulls = df.isna().sum()
    print(nulls[nulls > 0])
    save(df)
    return df

In [14]:
if __name__ == "__main__":
    df = main()

loaded UNSW-NB15_1.csv: (700001, 49)
loaded UNSW-NB15_2.csv: (700001, 49)
loaded UNSW-NB15_3.csv: (700001, 49)
loaded UNSW-NB15_4.csv: (440044, 49)
combined shape: (2540047, 49)
sport: 2 values not numeric -> -1. examples: {'-': 2}
dsport: 7 values not numeric -> -1. examples: {'-': 7}
ct_ftp_cmd: 1429879 blank/non-numeric -> 0
ct_flw_http_mthd: NaN in 3.9% of http rows, 57.4% of non-http rows
is_ftp_login: NaN in 54.3% of http rows, 56.5% of non-http rows
is_ftp_login values: {0: 2496472, 1: 43389, 4: 156, 2: 30}
Label vs attack_cat mismatches: 0
{'Normal': 2218764, 'Generic': 215481, 'Exploits': 44525, 'Fuzzers': 24246, 'DoS': 16353, 'Reconnaissance': 13987, 'Analysis': 2677, 'Backdoor': 2329, 'Shellcode': 1511, 'Worms': 174}
Stime range: 2015-01-22 11:49:37 -> 2015-02-18 12:21:08

rows and attack share per day:
               rows  attack_share
Stime                            
2015-01-22  1052863       0.02110
2015-01-23    34340       0.00000
2015-02-18  1452844       0.20585

nul